<a href="https://colab.research.google.com/github/williamfaraday123/SC4001-Neural-Network/blob/main/Lim_Isaac_Part_A_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question A4

In this section, we will understand the utility of such a neural network in real world scenarios.

#### Please use the real record data named ‘record.wav’  as a test sample. Preprocess the data using the provided preprocessing script (data_preprocess.ipynb) and prepare the dataset.
Do a model prediction on the sample test dataset and obtain the predicted label using a threshold of 0.5. The model used is the optimized pretrained model using the selected optimal batch size and optimal number of neurons.
Find the top 3 most important features on the model prediction for the test sample using SHAP. Plot the local feature importance with a force plot and explain your observations. (Refer to the documentation and these three useful references:
https://christophm.github.io/interpretable-ml-book/shap.html#examples-5,
https://towardsdatascience.com/deep-learning-model-interpretation-using-shap-a21786e91d16,  
https://medium.com/mlearning-ai/shap-force-plots-for-classification-d30be430e195)



1. Firstly, we import relevant libraries.

In [1]:
!wget https://raw.githubusercontent.com/williamfaraday123/SC4001-Neural-Network/refs/heads/main/common_utils.py -O common_utils.py

--2026-03-09 04:57:51--  https://raw.githubusercontent.com/williamfaraday123/SC4001-Neural-Network/refs/heads/main/common_utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3724 (3.6K) [text/plain]
Saving to: ‘common_utils.py’

common_utils.py     100%[===================>]   3.64K  --.-KB/s    in 0s      

2026-03-09 04:57:51 (41.8 MB/s) - ‘common_utils.py’ saved [3724/3724]



In [2]:
import tqdm
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from scipy.io import wavfile as wav

from sklearn import preprocessing
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
from common_utils import set_seed

# setting seed
set_seed()

To reduce repeated code, place your
network (MLP defined in QA1)
torch datasets (CustomDataset defined in QA1)
loss function (loss_fn defined in QA1)
in a separate file called common_utils.py

Import them into this file. You will not be repenalised for any error in QA1 here as the code in QA1 will not be remarked.

The following code cell will not be marked.


In [3]:
# YOUR CODE HERE
from common_utils import MLP, CustomDataset, initialise_loaders, split_dataset, preprocess_dataset, EarlyStopper
loss_fn = nn.BCELoss()

2. Install and import shap

In [4]:
# YOUR CODE HERE
import shap

3. Read the csv data preprocessed from 'record.wav', using variable name 'df', and fill the size of 'df' in 'size_row' and 'size_column'.

In [5]:
df = 0
size_row = 0
size_column = 0
# YOUR CODE HERE

df = pd.read_csv('https://raw.githubusercontent.com/williamfaraday123/SC4001-Neural-Network/refs/heads/main/new_record.csv')
size_row, size_column = df.shape

In [6]:
size_row, size_column

(1, 78)

 4.  Preprocess to obtain the test data, save the test data as numpy array.

In [8]:

def preprocess(X_train, df):
    """preprocess your dataset to obtain your test dataset, remember to remove the 'filename' as Q1
    """
    # YOUR CODE HERE

    # 1. Remove 'filename' as it is not a feature
    if 'filename' in df.columns:
        df_features = df.drop(columns=['filename'])
    else:
        df_features = df.copy()

    # 2. CLEANING: Convert string-lists like '[123.4]' to actual floats
    # This applies to every cell in the dataframe
    df_features = df_features.applymap(lambda x:
        float(x.strip('[]')) if isinstance(x, str) and '[' in x else x
    )

    # 2. Ensure X_train is a numpy array for the scaler
    X_train_np = X_train.to_numpy() if hasattr(X_train, 'to_numpy') else X_train
    X_test_np = df_features.to_numpy()

    # 3. Use the preprocess_dataset utility to scale based on X_train
    # This function returns (X_train_scaled, X_test_scaled)
    _, X_test_scaled_eg = preprocess_dataset(X_train_np, X_test_np)

    return X_test_scaled_eg

def preprocess(X_train, df):
    # 1. Remove 'filename'
    X_new_raw = df.drop(columns=['filename'])

    # 2. CLEANING: Convert string-lists like '[123.4]' to actual floats
    # This applies to every cell in the dataframe
    X_new_raw = X_new_raw.applymap(lambda x:
        float(x.strip('[]')) if isinstance(x, str) and '[' in x else x
    )

    # 3. Ensure X_train is also cleaned if it came from the same source
    # (If X_train is already numeric, this won't hurt)
    X_train_clean = X_train.applymap(lambda x:
        float(x.strip('[]')) if isinstance(x, str) and '[' in x else x
    )

    # 4. Scale the record
    _, X_test_scaled_eg = preprocess_dataset(X_train_clean.to_numpy(), X_new_raw.to_numpy())

    return X_test_scaled_eg

# --- STEP 1: Get X_train from the original data split ---
df_original = pd.read_csv('https://raw.githubusercontent.com/williamfaraday123/SC4001-Neural-Network/refs/heads/main/simplified.csv')
df_original['label'] = df_original['filename'].apply(lambda x: 'positive' if 'pos' in x.lower() else 'negative')

X_train, y_train, X_test, y_test = split_dataset(
    df_original,
    columns_to_drop=['filename', 'label'],
    test_size=0.25,
    random_state=42
)

X_test_scaled_eg = preprocess(X_train, df)

/tmp/ipykernel_19455/2962054090.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  X_new_raw = X_new_raw.applymap(lambda x:
/tmp/ipykernel_19455/2962054090.py:40: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  X_train_clean = X_train.applymap(lambda x:


5. Do a model prediction on the sample test dataset and obtain the predicted label using a threshold of 0.5. The model used is the optimized pretrained model using the selected optimal learning rate and optimal architecture. Note: Please define the variable of your final predicted label as 'pred_label'.

In [ ]:
# YOUR CODE HERE

# 1. Ensure the model is in evaluation mode
final_model.eval()

# 2. Convert the processed record to a PyTorch tensor
X_test_tensor = torch.tensor(X_test_scaled_eg, dtype=torch.float32)

# 3. Perform the forward pass
with torch.no_grad():
    probability = final_model(X_test_tensor)

# 4. Apply the 0.5 threshold to get the final predicted label
# .item() extracts the value if it's a single record
pred_label = (probability > 0.5).float().item()

# Map back to string for clarity (Optional)
result_text = "Positive Polarity" if pred_label == 1 else "Negative Polarity"

print(f"Predicted Class: {int(pred_label)} ({result_text})")
print(f"Confidence Score: {probability.item():.4f}")

6. Find the most important features on the model prediction for your test sample using SHAP. Create an instance of the DeepSHAP which is called DeepExplainer using traianing dataset: https://shap.readthedocs.io/en/latest/generated/shap.DeepExplainer.html.

Plot the local feature importance with a force plot and explain your observations.  (Refer to the documentation and these three useful references:
https://christophm.github.io/interpretable-ml-book/shap.html#examples-5,
https://towardsdatascience.com/deep-learning-model-interpretation-using-shap-a21786e91d16,  
https://medium.com/mlearning-ai/shap-force-plots-for-classification-d30be430e195)


In [ ]:
'''
Fit the explainer on a subset of the data (you can try all but then gets slower)
Return approximate SHAP values for the model applied to the data given by X.
Plot the local feature importance with a force plot and explain your observations.
'''
# YOUR CODE HERE